# 9 - Contract Reviewer
## Legal Document Critique + Citation-Grounded Risk Register
20 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent-use-cases/blob/main/examples/9-contract-reviewer/contract_reviewer_workbook.ipynb)

Every lawyer knows the pain: you receive a 40-page contract, you need to find every clause that creates risk, flag missing protections, and build a negotiation list -- in 2 hours, before the client meeting.

This example builds a contract review agent with one critical constraint: **every risk finding must cite the clause it comes from**. No clause reference = no finding. This forces the model to ground its analysis in the actual document rather than generating generic legal boilerplate.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | The citation problem -- why grounding matters in legal review |
| 2 | Schema design -- RiskFinding, MissingProtection, NegotiationPoint |
| 3 | The citation-mandatory system prompt |
| 4 | Review a Services Agreement |
| 5 | Review an NDA |
| 6 | Compare: with vs without citation enforcement |
| * | Exercises + Answer Key |

### Prerequisites
- Python 3.10+, or Google Colab
- OPENAI_API_KEY in .env or Colab Secrets

In [ ]:
try:
    from google.colab import userdata  # noqa: F401
    import subprocess
    subprocess.run(["pip", "install", "-q", "langchain-openai", "pydantic", "python-dotenv"], check=True)
    print("Packages installed.")
except ImportError:
    print("Local environment.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    if not os.getenv("OPENAI_API_KEY"):
        raise EnvironmentError("Set OPENAI_API_KEY in .env or Colab Secrets.")
print("API key ready.")

---

## Part 1 - The Citation Problem

Ask a generic LLM to review a contract and you get output like this:

> "The liability clause may expose you to unlimited risk. You should negotiate a cap."

That is useless. Which liability clause? Section 5? Section 12? What exactly does it say?

The citation-grounded pattern enforces specificity:

| Without citation | With citation |
|-----------------|---------------|
| "The liability clause is risky" | "Section 5 imposes unlimited liability on Service Provider while capping Client liability at zero" |
| Generic boilerplate | Clause-specific, actionable |
| Lawyer cannot use it | Lawyer can act on it immediately |

The system prompt does the enforcement: **"If you cannot point to a specific clause, do not include that finding."**

---

## Part 2 - Schema Design

In [ ]:
from typing import List, Literal, Optional
from pydantic import BaseModel, Field


class RiskFinding(BaseModel):
    severity: Literal["critical", "high", "medium", "low"]
    category: Literal[
        "liability", "payment", "ip", "termination",
        "confidentiality", "governance", "compliance", "other"
    ]
    clause_reference: str = Field(
        description="Section or clause number, e.g. 'Section 8.2'"
    )
    issue: str = Field(description="One sentence describing what creates the risk")
    implication: str = Field(description="Business impact if this risk materializes")
    recommended_redline: str = Field(
        description="Specific language change to address this risk"
    )


class MissingProtection(BaseModel):
    protection: str = Field(description="Standard protection absent from this contract")
    why_needed: str = Field(description="Why this protection matters for this contract type")
    suggested_clause: str = Field(description="Draft language to add")


class NegotiationPoint(BaseModel):
    priority: Literal["must_have", "should_have", "nice_to_have"]
    topic: str
    current_position: str = Field(description="What the contract currently says")
    target_position: str = Field(description="What you should push for")


class ContractReview(BaseModel):
    contract_type: str
    counterparty: Optional[str] = None
    governing_law: Optional[str] = None
    overall_risk: Literal["high", "medium", "low"]
    executive_summary: str = Field(
        description="2-3 sentence summary for a senior executive"
    )
    risk_findings: List[RiskFinding]
    missing_protections: List[MissingProtection]
    negotiation_points: List[NegotiationPoint]

print("Schema defined.")
print(f"RiskFinding fields: {list(RiskFinding.model_fields.keys())}")

---

## Part 3 - The Citation-Mandatory System Prompt

The system prompt has one job beyond instructing the model: it enforces a rule that **cannot be expressed in the schema alone**.

The schema says `clause_reference: str` -- but a string can be empty or vague ("somewhere in section 3").
The prompt says: *"If you cannot point to a specific clause, do not include that finding."*

This turns citation from a suggestion into a gate.

In [ ]:
from langchain_core.messages import SystemMessage
from langchain_openai import ChatOpenAI

REVIEWER_SYSTEM = SystemMessage(
    """You are a senior commercial lawyer reviewing a contract on behalf of a client.

Your analysis must cover three areas:

1. RISK FINDINGS -- every clause that creates risk for the client
   - You MUST include a clause_reference (section/clause number) for every finding
   - If you cannot point to a specific clause, do not include that finding
   - Quote or closely paraphrase the language that creates the risk
   - recommended_redline must be concrete proposed language, not vague advice

2. MISSING PROTECTIONS -- standard clauses absent from this contract
   - Only flag protections genuinely standard for this contract type
   - Provide draft suggested_clause language the client can propose

3. NEGOTIATION POINTS -- prioritised list of what to push for
   - must_have: deal-breakers; walk away if not addressed
   - should_have: important but not fatal
   - nice_to_have: improvements worth raising if counterparty is cooperative

Be thorough but precise. A finding without a clause reference is worthless."""
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
reviewer = REVIEWER_SYSTEM | llm.with_structured_output(ContractReview)
print("Reviewer ready.")

---

## Part 4 - Review a Services Agreement

This services agreement has several deliberately adversarial clauses:
- 90-day payment terms with no late interest
- Unlimited liability on the vendor, zero on the client
- IP assignment that sweeps in pre-existing tools
- Asymmetric termination rights

In [ ]:
SERVICES_AGREEMENT = """
PROFESSIONAL SERVICES AGREEMENT

This Professional Services Agreement is entered into between Acme Corp (Client) and Vendor Ltd (Service Provider).

Section 1. Services
Service Provider shall perform consulting services as directed by Client from time to time.
Client may modify the scope of services at any time without notice.

Section 2. Payment
Client shall pay Service Provider within 90 days of invoice receipt. Late payments shall
not accrue interest. Client may withhold payment for any reason it deems appropriate.

Section 3. Intellectual Property
All work product, inventions, and developments created by Service Provider under this
Agreement shall be the sole property of the Client. Service Provider hereby assigns all
right, title, and interest in such work product to Client. This assignment includes all
pre-existing tools and methodologies used in the delivery.

Section 4. Confidentiality
Service Provider agrees to keep all Client information confidential for a period of 1 year
following termination of this Agreement.

Section 5. Liability
In no event shall Client be liable to Service Provider for any damages, whether direct,
indirect, incidental, special, or consequential. Service Provider's liability to Client shall be unlimited.

Section 6. Termination
Client may terminate this Agreement immediately and without cause at any time.
Service Provider may not terminate this Agreement without 180 days written notice.

Section 7. Governing Law
This Agreement shall be governed by the laws of the State of Delaware.
"""

review_sa = reviewer.invoke(f"CONTRACT TO REVIEW:\n\n{SERVICES_AGREEMENT}")

SEV = {"critical": "CRIT", "high": "HIGH", "medium": "MED ", "low": "LOW "}
PRI = {"must_have": "!!!", "should_have": "!! ", "nice_to_have": "!  "}

print(f"Type: {review_sa.contract_type}")
print(f"Overall risk: {review_sa.overall_risk.upper()}")
print(f"\nExecutive Summary:\n{review_sa.executive_summary}")
print(f"\nRisk Findings ({len(review_sa.risk_findings)}):")
for f in review_sa.risk_findings:
    print(f"  [{SEV[f.severity]}] [{f.clause_reference}] {f.issue}")
print(f"\nEvery finding has clause_reference: {all(f.clause_reference for f in review_sa.risk_findings)}")
print(f"Return type: {type(review_sa).__name__}")

---

## Part 5 - Review an NDA

In [ ]:
NDA = """
NON-DISCLOSURE AGREEMENT

This Non-Disclosure Agreement is entered into between DisclosedCo and RecipientCo.

1. Confidential Information
Confidential Information means all information disclosed by Disclosing Party that is
marked as confidential or that a reasonable person would understand to be confidential.

2. Obligations
Recipient shall hold Confidential Information in strict confidence and shall not disclose
it to any third party. Recipient may share Confidential Information with employees who
need to know it for the Purpose.

3. Term
This Agreement shall remain in effect for 5 years from the Effective Date. Obligations
with respect to trade secrets shall survive indefinitely.

4. Return of Information
Upon request, Recipient shall destroy or return all Confidential Information.

5. Governing Law
This Agreement shall be governed by English law.
"""

review_nda = reviewer.invoke(f"CONTRACT TO REVIEW:\n\n{NDA}")

print(f"Type: {review_nda.contract_type}")
print(f"Overall risk: {review_nda.overall_risk.upper()}")
print(f"\nRisk Findings ({len(review_nda.risk_findings)}):")
for f in review_nda.risk_findings:
    print(f"  [{SEV[f.severity]}] [{f.clause_reference}] {f.issue}")
print(f"\nMissing Protections ({len(review_nda.missing_protections)}):")
for p in review_nda.missing_protections:
    print(f"  - {p.protection}")
print(f"\nNegotiation Points ({len(review_nda.negotiation_points)}):")
for n in review_nda.negotiation_points:
    print(f"  [{PRI[n.priority]}] {n.topic}")

---

## Part 6 - Compare: With vs Without Citation Enforcement

In [ ]:
PERMISSIVE_SYSTEM = SystemMessage(
    """You are a senior commercial lawyer reviewing a contract.
Identify risks, missing protections, and negotiation points."""
)

permissive_reviewer = PERMISSIVE_SYSTEM | llm.with_structured_output(ContractReview)
review_permissive = permissive_reviewer.invoke(f"CONTRACT TO REVIEW:\n\n{NDA}")

print("WITHOUT citation enforcement:")
for f in review_permissive.risk_findings:
    ref = f.clause_reference or "(empty)"
    print(f"  clause_reference: '{ref}'")

print()
print("WITH citation enforcement:")
for f in review_nda.risk_findings:
    print(f"  clause_reference: '{f.clause_reference}'")

---

## Exercise 1 - Add a Data Privacy Compliance Check

Extend `ContractReview` to include a `privacy_flags` field.

Add a `PrivacyFlag` schema with:
- `regulation`: which law applies (GDPR, CCPA, PDPA, etc.)
- `gap`: what is missing or non-compliant
- `clause_reference`: must cite a clause, or "absent" if missing entirely

In [ ]:
# Exercise 1 Starter
class PrivacyFlag(BaseModel):
    regulation: str = Field(description="e.g. 'GDPR', 'CCPA', 'PDPA'")
    gap: str = Field(description="What is missing or non-compliant")
    clause_reference: str = Field(description="Clause number or 'absent' if missing entirely")

# TODO: Add privacy_flags: List[PrivacyFlag] to a new ContractReviewWithPrivacy class
# TODO: Update the system prompt to include data privacy check instructions
# TODO: Run on SERVICES_AGREEMENT and inspect privacy_flags

In [ ]:
# Exercise 1 Answer Key
class ContractReviewWithPrivacy(BaseModel):
    contract_type: str
    counterparty: Optional[str] = None
    governing_law: Optional[str] = None
    overall_risk: Literal["high", "medium", "low"]
    executive_summary: str
    risk_findings: List[RiskFinding]
    missing_protections: List[MissingProtection]
    negotiation_points: List[NegotiationPoint]
    privacy_flags: List[PrivacyFlag] = Field(
        default_factory=list,
        description="Data privacy compliance gaps"
    )

PRIVACY_SYSTEM = SystemMessage(
    """You are a senior commercial lawyer with data privacy expertise.

Review the contract and include:
1. Standard risk findings (cite clause reference for each)
2. Missing protections standard for this contract type
3. Negotiation priorities
4. DATA PRIVACY FLAGS -- check for:
   - Data processing terms (who processes what, on what basis)
   - Data retention limits
   - Data subject rights obligations
   - Cross-border transfer restrictions
   For each gap, cite the clause where it should appear or mark 'absent'."""
)

privacy_reviewer = PRIVACY_SYSTEM | llm.with_structured_output(ContractReviewWithPrivacy)
review_privacy = privacy_reviewer.invoke(f"CONTRACT TO REVIEW:\n\n{SERVICES_AGREEMENT}")

print(f"Privacy flags ({len(review_privacy.privacy_flags)}):")
for flag in review_privacy.privacy_flags:
    print(f"  [{flag.regulation}] {flag.gap} (ref: {flag.clause_reference})")

---

## What You Built

A contract review agent that enforces citation discipline through the system prompt.

The schema alone cannot enforce grounding -- `clause_reference: str` accepts any string.
The prompt completes the contract: *"If you cannot point to a specific clause, do not include that finding."*

This pattern generalises to any domain where assertions must be traceable:
- Regulatory compliance checks (next example)
- Audit findings with evidence requirements
- Medical literature review with source citation
- Financial disclosure review with filing references

---
*Example 9 of 23 - agent-use-cases*